# 🎓 Difficulty-Controlled Question Generation Experiment

**Complete Pipeline for Educational QG Research**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/thesis-project/blob/paper-implementation/notebooks/QG_Experiment_Colab.ipynb)

---

## 📋 What This Notebook Does

1. ✅ Setup environment with GPU
2. ✅ Fetch SQuAD dataset
3. ✅ Add difficulty labels
4. ✅ Train baseline & controlled models
5. ✅ Generate & evaluate questions
6. ✅ Create paper-ready results

⏱️ **Runtime**: 2-3 hours with GPU  
💾 **Enable GPU**: Runtime → Change runtime type → GPU

## 🔧 Step 1: Setup

In [3]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Clone repository
!git clone https://github.com/YOUR_USERNAME/thesis-project.git
%cd thesis-project
!git checkout paper-implementation

In [ ]:
# Install dependencies
!pip install -q transformers>=4.44 datasets>=2.20 evaluate rouge-score sacrebleu sentencepiece accelerate openpyxl matplotlib seaborn scikit-learn pyyaml

##📊 Step 2: Data Preparation

In [ ]:
# Fetch SQuAD
!python -m src.data.fetch_squad --out data/raw/squad.jsonl

In [ ]:
# Create subset
!python -m src.data.make_subset --in data/raw/squad.jsonl --out data/interim/squad_small.jsonl --train 1000 --dev 200 --test 200 --seed 42

In [ ]:
# Add difficulty labels
!python -m src.data.preprocess --in data/interim/squad_small.jsonl --out data/processed/squad_small.qg_labeled.jsonl

## 🎯 Step 3: Training

In [ ]:
# Train baseline
!python -m src.train.finetune_baseline --cfg src/config/exp_squad_small.yaml

In [ ]:
# Train controlled
!python -m src.train.finetune_controlled --cfg src/config/exp_squad_small_controlled.yaml

## 💬 Step 4: Generation

In [ ]:
# Generate - Baseline
!python -m src.generate.run_generate --model outputs/baseline_t5_small_squad_small --dataset data/processed/squad_small.qg_labeled.jsonl --split test --out reports/tables/preds_baseline.jsonl

In [ ]:
# Generate - Controlled
!python -m src.generate.run_generate --model outputs/controlled_t5_small_squad_small --dataset data/processed/squad_small.qg_labeled.jsonl --split test --out reports/tables/preds_controlled.jsonl

## 📈 Step 5: Evaluation

In [ ]:
# BLEU/ROUGE
!python -m src.eval.compute_bleu_rouge --pred reports/tables/preds_baseline.jsonl --out reports/tables/metrics_baseline.json
!python -m src.eval.compute_bleu_rouge --pred reports/tables/preds_controlled.jsonl --out reports/tables/metrics_controlled.json

In [ ]:
# QA Answerability
!python -m src.eval.qa_answerability --dataset data/processed/squad_small.qg_labeled.jsonl --pred reports/tables/preds_baseline.jsonl --out reports/tables/qa_baseline.json
!python -m src.eval.qa_answerability --dataset data/processed/squad_small.qg_labeled.jsonl --pred reports/tables/preds_controlled.jsonl --out reports/tables/qa_controlled.json

In [ ]:
# Aggregate results
!python -m src.eval.aggregate --metricsA reports/tables/metrics_baseline.json --metricsB reports/tables/metrics_controlled.json --qaA reports/tables/qa_baseline.json --qaB reports/tables/qa_controlled.json --out_csv reports/tables/comparison.csv --out_fig reports/figures/comparison.png --emit_latex paper/results.tex

## 📊 Step 6: View Results

In [ ]:
# Show metrics
import json
import pandas as pd
from IPython.display import Image, display

with open('reports/tables/metrics_baseline.json') as f:
    baseline = json.load(f)
with open('reports/tables/metrics_controlled.json') as f:
    controlled = json.load(f)

df = pd.DataFrame({
    'Metric': ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L'],
    'Baseline': [baseline['bleu'], baseline['rouge1'], baseline['rouge2'], baseline['rougeL']],
    'Controlled': [controlled['bleu'], controlled['rouge1'], controlled['rouge2'], controlled['rougeL']]
})
print(df)

display(Image('reports/figures/comparison.png'))

## 📥 Step 7: Download Results

In [ ]:
# Download files
from google.colab import files
files.download('reports/tables/comparison.csv')
files.download('reports/figures/comparison.png')
files.download('paper/results.tex')

## ✅ Complete!

You now have:
- Trained models
- Generated questions
- Evaluation metrics
- LaTeX tables for your paper

Next: Complete human evaluation and include results.tex in your paper!